# QFIS — QLoRA Fine-tuning on Google Colab
**Model:** microsoft/phi-2 (2.7B) | **Method:** QLoRA 4-bit NF4 | **Dataset:** FinQA

### Before running:
- **Runtime → Change runtime type → T4 GPU** ✅
- Run cells **one by one** in order
- After last cell download `qfis_model_v1.zip` and put contents in `models/v1/`

## Step 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
import sys
print(f'Python: {sys.version}')

## Step 2 — Install Correct Versions (fixes bitsandbytes + triton error)

In [ ]:
# Uninstall conflicting versions first
!pip uninstall -y bitsandbytes triton 2>/dev/null

# Install pinned compatible versions for Colab T4 (CUDA 12.x)
!pip install -q \
    bitsandbytes==0.45.5 \
    transformers==4.47.0 \
    peft==0.14.0 \
    accelerate==1.3.0 \
    datasets==3.2.0 \
    einops \
    loguru \
    scikit-learn

# Restart runtime to apply new packages cleanly
print('Packages installed. Now click Runtime → Restart session, then continue from Step 3.')

## ⚠️ RESTART RUNTIME NOW
After Step 2 finishes → **Runtime → Restart session** → then run Step 3 onwards (skip Step 1 and 2)

## Step 3 — Verify bitsandbytes GPU support

In [ ]:
import bitsandbytes as bnb
import torch
print(f'bitsandbytes version: {bnb.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
# Should print version 0.45.x and CUDA: True
# If you see 'compiled without GPU support' error, re-run Step 2 and restart again

## Step 4 — Load and Preprocess FinQA Dataset

In [ ]:
import json, re, hashlib
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def normalize_answer(text):
    text = str(text).lower().strip()
    text = re.sub(r'(\d),(\d)', r'\1\2', text)
    return re.sub(r'\s+', ' ', text).strip()

def build_context(r):
    parts = []
    for field in ['pre_text', 'post_text']:
        val = r.get(field, [])
        if isinstance(val, list):
            parts.extend([p.strip() for p in val if isinstance(p, str) and p.strip()])
        elif isinstance(val, str) and val.strip():
            parts.append(val.strip())
    for row in r.get('table', []):
        if isinstance(row, list):
            parts.append(' | '.join(str(c) for c in row))
    return ' '.join(parts)[:1000]

def build_prompt(q, ctx):
    return f'Context: {ctx}\n\nQuestion: {q}\n\nAnswer:'

print('Loading FinQA from HuggingFace...')
ds = load_dataset('dreamerdeo/finqa', trust_remote_code=True)

records = []
for split in ['train', 'validation', 'test']:
    if split in ds:
        records.extend([dict(x) for x in ds[split]])
print(f'Raw: {len(records)} records')

# Clean
cleaned = []
for r in records:
    q = str(r.get('question', '')).strip()
    a = normalize_answer(r.get('answer', ''))
    ctx = build_context(r)
    if len(q) < 5 or not a or a in ('none','n/a','','nan','null') or len(ctx) < 10:
        continue
    cleaned.append({'question': q, 'answer': a, 'context': ctx,
                    'prompt': build_prompt(q, ctx), 'completion': a})

# Deduplicate
seen, unique = set(), []
for r in cleaned:
    key = hashlib.md5(r['question'].encode()).hexdigest()
    if key not in seen:
        seen.add(key)
        unique.append(r)

# 70/15/15 split
train_data, temp = train_test_split(unique, test_size=0.30, random_state=42)
val_data, test_data = train_test_split(temp, test_size=0.50, random_state=42)

# T4 has 15GB — use 2000 train samples for good quality + reasonable time
train_data = train_data[:2000]
val_data   = val_data[:400]

# Save test set for evaluation later
with open('/content/test_data.json', 'w') as f:
    json.dump(test_data[:300], f)

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test saved: {len(test_data)}')
print('Sample:', train_data[0]['prompt'][:150], '...')

## Step 5 — Load Phi-2 with 4-bit Quantization

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = 'microsoft/phi-2'
MAX_SEQ_LEN = 384

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, trust_remote_code=True, padding_side='right'
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading Phi-2 with 4-bit NF4...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Phi-2 loaded successfully!')

## Step 6 — Apply LoRA

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'dense'],
    bias='none',
    inference_mode=False,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~10M trainable / 2.7B total (~0.37%)

## Step 7 — Tokenize Dataset

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

def build_hf_dataset(records):
    texts = [r['prompt'] + ' ' + r['completion'] + tokenizer.eos_token
             for r in records]
    def tokenize(batch):
        enc = tokenizer(batch['text'], truncation=True,
                        max_length=MAX_SEQ_LEN, padding='max_length')
        enc['labels'] = enc['input_ids'].copy()
        return enc
    ds = Dataset.from_dict({'text': texts})
    return ds.map(tokenize, batched=True, remove_columns=['text'])

print('Tokenizing...')
train_ds = build_hf_dataset(train_data)
val_ds   = build_hf_dataset(val_data)
print(f'Done — Train: {len(train_ds)} | Val: {len(val_ds)}')

## Step 8 — Train (~25-35 min on T4)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    dataloader_num_workers=2,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Training started...')
trainer.train()
print('Training complete!')

## Step 9 — Save Weights

In [ ]:
import json, os

SAVE_DIR = '/content/qfis_model_v1'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f'{SAVE_DIR}/training_meta.json', 'w') as f:
    json.dump({
        'base_model': BASE_MODEL,
        'lora_r': 16, 'lora_alpha': 32,
        'epochs': 3, 'max_seq_len': MAX_SEQ_LEN,
        'quantization': '4-bit NF4',
        'train_samples': len(train_data),
        'trained_on': 'Google Colab T4',
    }, f, indent=2)

print('Saved files:', os.listdir(SAVE_DIR))

## Step 10 — Quick Test (verify model works before downloading)

In [ ]:
model.eval()
test_prompt = 'Context: The company reported total revenue of $5.2 billion in 2022, up from $4.8 billion in 2021.\n\nQuestion: What was the total revenue in 2022?\n\nAnswer:'
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
new_tokens = out[0][inputs['input_ids'].shape[1]:]
print('Answer:', tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
print('Expected: something like "5.2 billion" or "$5.2 billion"')

## Step 11 — Zip and Download

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/qfis_model_v1', 'zip', '/content/qfis_model_v1')
print('Zip created. Starting download...')
files.download('/content/qfis_model_v1.zip')
print('Done! Extract and place all files into models/v1/ in your project.')

---
## After Download — What to do locally
```
1. Extract qfis_model_v1.zip
2. Copy ALL files into:  models/v1/
3. Restart backend:      uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
4. Run evaluation:       python training/evaluate.py   (from backend/ folder)
```
Files you should see in models/v1/:
- adapter_config.json  ← confirms fine-tuned model loaded
- adapter_model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_meta.json